<a href="https://colab.research.google.com/github/smduarte/spbd-2627/blob/main/docs/labs/lab4/SPBD_Labs_spark1_exercises_solutions.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lab3 - Python Spark Exercises


In [ ]:
#@title Download "Os Maias"
!wget -q -O os_maias.txt https://www.dropbox.com/s/n24v0z7y79np319/os_maias.txt?dl=0

##1. Word Frequency Exercises

1.1) Create a [Spark Core](https://spark.apache.org/docs/latest/api/python/reference/pyspark.html) program that counts the number of occurrences of each word in the novel "Os Maias":

a) No sorting required, words and frequency can appear in any order;

b) Sorted by word, in reverse alphabetical order;

c) Sorted by frequency (the words with higher occurrence first).

Note that the sorting should be performed as a transformation (i.e. it should produce an RDD)...

In [ ]:
#@title Exercise 1.1a - WordFrequency: No Sorting
import re, pyspark
from pyspark.sql import SparkSession
from pyspark.sql.functions import *

spark = SparkSession \
    .builder \
    .appName("WordFrequency1.1a") \
    .getOrCreate()

sc = spark.sparkContext

try:
  lines = sc.textFile('os_maias.txt') \
      .map( lambda line: line.strip().lower() ) \

  words = lines.flatMap( lambda line: re.findall(r"[^\W_]+", line) )

  frequencies = words.map( lambda word: (word, 1)) \
                .reduceByKey( lambda a, b: a + b)


  for w,f in frequencies.take(20):
      print("{}\t{}".format(w,f))

except Exception as e:
  print(e)

In [ ]:
#@title Exercise 1.1b - WordFrequency: Sorted by word, alphabetically

import re, pyspark
from pyspark.sql import SparkSession
from pyspark.sql.functions import *

spark = SparkSession \
    .builder \
    .appName("WordFrequency1.1a") \
    .getOrCreate()

sc = spark.sparkContext

try:
  lines = sc.textFile('os_maias.txt') \
      .map( lambda line: line.strip().lower() ) \

  words = lines.flatMap( lambda line: re.findall(r"[^\W_]+", line) )

  frequencies = words.map( lambda word: (word, 1)) \
                .reduceByKey( lambda a, b: a + b) \
                .sortByKey()


  for w,f in frequencies.take(20):
      print("{}\t{}".format(w,f))

except Exception as e:
  print(e)

In [ ]:
#@title Exercise 1.1c - WordFrequency: Sorted by Frequency
import re, pyspark
from pyspark.sql import SparkSession
from pyspark.sql.functions import *

spark = SparkSession \
    .builder \
    .appName("WordFrequency1.1a") \
    .getOrCreate()

sc = spark.sparkContext

try:
  lines = sc.textFile('os_maias.txt') \
      .map( lambda line: line.strip().lower() ) \

  words = lines.flatMap( lambda line: re.findall(r"[^\W_]+", line) )

  frequencies = words.map( lambda word: (word, 1)) \
                .reduceByKey( lambda a, b: a + b) \
                .sortBy( lambda kv : -kv[1])


  for w,f in frequencies.take(20):
      print("{}\t{}".format(w,f))

except Exception as e:
  print(e)

1.2) Create a Spark program that the top 10 most used words in "Os Maias" novel.

You should try to avoid sorting or finding the top-10 as actions. Your top-10 most used words should be a RDD at the end of the computation. Check *zipWithIndex* in [pyspark RDD](https://spark.apache.org/docs/latest/api/python/reference/api/pyspark.RDD.html#pyspark.RDD) documentation

In [ ]:
#@title Exercise 1.2 - Top 10 Words by frequency

import pyspark, re
from pyspark.sql import SparkSession

spark = SparkSession \
    .builder \
    .appName("Top10Words 1.2)") \
    .getOrCreate()

sc = spark.sparkContext

try:
  lines = sc.textFile('os_maias.txt') \
      .map( lambda line: line.strip() ) \

  words = lines.flatMap( lambda line: re.findall(r"[^\W_]+", line) )

  top10_words_partitions = words.map( lambda word: (word, 1)) \
          .reduceByKey( lambda a, b: a + b) \
          .mapPartitions( lambda partition : sorted(partition, key=lambda kv: kv[1], reverse=True)[0:10])

  top10_words = top10_words_partitions.sortBy(lambda x: x[1], ascending = False) \
                .zipWithIndex() \
                .filter( lambda ranked: ranked[1] < 10) \
                .map( lambda ranked: ranked[0])

  print("Partitions: Top-10")
  for x in top10_words_partitions.glom().collect():
      print("{}".format(x))

  print("Top-10 Most frequent words:")
  for x in top10_words.collect():
      print("{}".format(x))

except Exception as e:
  print(e)


Partitions: Top-10
[('o', 6936), ('um', 3026), ('da', 2202), ('uma', 2170), ('para', 1737), ('E', 1698), ('lhe', 1562), ('no', 1478), ('as', 1426), ('ao', 1394)]
[('de', 8357), ('a', 6945), ('que', 4986), ('e', 4533), ('se', 3041), ('com', 2794), ('do', 2571), ('os', 1808), ('Carlos', 1797), ('não', 1661)]
Top-10 Most frequent words:
('de', 8357)
('a', 6945)
('o', 6936)
('que', 4986)
('e', 4533)
('se', 3041)
('um', 3026)
('com', 2794)
('do', 2571)
('da', 2202)


##2. Weblog Analysis

Consider a set of log files captured during a DDOS (*Distributed Denial of Service*) attack, containing information for the web accesses performed during the attack to the server.

The log files contain text lines as shown below, with TAB as the separator:

date |IP_source | status_code | operation | URL | execution time |
-|-|-|-|-|-
timestamp  | string | int | string | string| float |
2016-12-06T08:58:35.318+0000|37.139.9.11|404|GET|/codemove/TTCENCUFMH3C|0.026

In [ ]:
#@title Download the dataset
!wget -q -O weblog.txt https://www.dropbox.com/s/0r8902uj9yum7dg/web.log?dl=0
!head -3 weblog.txt

2016-12-06T08:58:35.318+0000 37.139.9.11 404 GET /codemove/TTCENCUFMH3C 0.026  
2016-12-06T08:58:35.356+0000 178.22.148.122 404 GET /codemove/PSO83TYKET12 0.088  
2016-12-06T08:58:35.357+0000 178.22.148.122 404 GET /codemove/PSO83TYKET12 0.088  


2.1. Count the number of unique IP addresses involved in the DDOS attack.


In [ ]:
#@title Exercise 2.1 - Weblog : Unique IPs v1 (MapReduce inspired...)

import pyspark
from pyspark.sql import SparkSession

spark = SparkSession \
    .builder \
    .appName("Weblog") \
    .getOrCreate()

sc = spark.sparkContext

try:
  lines = sc.textFile('weblog.txt') \
          .map( lambda line: line.strip() )

  unique_ips = lines.map( lambda line: line.split(' ')) \
          .filter( lambda values: len(values) == 6) \
          .map( lambda values : (values[1], None)) \
          .reduceByKey( lambda a, b : None ) \
          .map( lambda _ : (None, 1)) \
          .reduceByKey( lambda a, b : a+b ) \


  for _,c in unique_ips.collect():
      print("{}".format(c))

except Exception as e:
  print(e)

167


In [ ]:
#@title Exercise 2.1 - Weblog : Unique IPs v2 ( using .distinct() )

import pyspark
from pyspark.sql import SparkSession

spark = SparkSession \
    .builder \
    .appName("Weblog") \
    .getOrCreate()

sc = spark.sparkContext

try:
  lines = sc.textFile('weblog.txt') \
          .map( lambda line: line.strip() )

  unique_ips = lines.map( lambda line: line.split(' ')) \
          .filter( lambda values: len(values) == 6) \
          .map( lambda values : values[1]) \
          .distinct()

  print(f"Unique Ips: {unique_ips.count()}")

except Exception as e:
  print(e)

Unique Ips: 167


2.2. For each interval of 10 seconds, provide the following information: [number of requests, average execution time, maximum time, minimum time]

In [ ]:
#@title Exercise 2.2 - Weblog : 10-second interval stats

import pyspark
from pyspark.sql import SparkSession

spark = SparkSession \
    .builder \
    .appName("Weblog") \
    .getOrCreate()

sc = spark.sparkContext

try:
  lines = sc.textFile('weblog.txt') \
         .map( lambda line: line.strip() )

  intervals = lines.map( lambda line: line.split(' ')) \
          .filter( lambda parts: len(parts) == 6) \
          .map( lambda parts: (parts[0][0:18], float(parts[5]))) \
          .map( lambda kv : (kv[0], (1, kv[1], kv[1], kv[1]))) \
          .reduceByKey( lambda a, b : (a[0] + b[0], a[1] + b[1], max(a[2],b[2]), min(a[3],b[3])) ) \
          .map( lambda kv : (kv[0], (kv[1][0], kv[1][1] / kv[1][0], kv[1][2], kv[1][3]))) \
          .sortByKey()

  for interval, stats in intervals.take(10):
    print(interval, stats)

except Exception as e:
  print(e)

2016-12-06T08:58:3 (483, 7.5934244306418215, 46.849, 0.013)
2016-12-06T08:58:4 (2611, 30.159845653006503, 69.654, 0.014)
2016-12-06T08:58:5 (5500, 38.52511163636371, 80.846, 0.017)
2016-12-06T08:59:0 (6914, 38.53438212322824, 81.659, 0.018)
2016-12-06T08:59:1 (6271, 32.96384978472328, 83.993, 0.017)
2016-12-06T08:59:2 (5434, 17.29333143172616, 77.967, 0.051)
2016-12-06T08:59:3 (8015, 11.210152214597631, 67.441, 0.056)
2016-12-06T08:59:4 (7947, 7.761815779539431, 65.706, 0.914)
2016-12-06T08:59:5 (5983, 3.821664382416849, 54.29, 0.678)
2016-12-06T09:00:0 (6882, 8.649971519907023, 45.314, 0.017)


2.3. Create an inverted index that, for each interval of 10 seconds, has a list of (unique) IPs executing accesses (to each URL).

In [ ]:
#@title Exercise 2.3 - Weblog : 10-second inverted index

import pyspark
from pyspark.sql import SparkSession

spark = SparkSession \
    .builder \
    .appName("Weblog") \
    .getOrCreate()

sc = spark.sparkContext

try:
  lines = sc.textFile('weblog.txt') \
         .map( lambda line: line.strip() )

  intervals = lines.map( lambda line: line.split(' ')) \
          .filter( lambda parts: len(parts) == 6) \
          .map( lambda parts: (f"{parts[0][0:18]}-{parts[4]}", { parts[1] } )) \
          .reduceByKey( lambda a, b : a | b)

  for interval, ips in intervals.take(10):
    print(interval, ips)

except Exception as e:
  print(e)


2016-12-06T08:58:3-/codemove/PSO83TYKET12 {'178.22.148.122'}
2016-12-06T08:58:3-/codemove/1U6HCG3V2S9D {'185.28.193.95'}
2016-12-06T08:58:3-/codemove/B35MFVKMU1C4 {'2002:894a:3a93:d:250:56ff:fe00:88c0'}
2016-12-06T08:58:3-/codemove/2CEBGK8M78Y7 {'192.241.151.220'}
2016-12-06T08:58:3-/codemove/ZBOWM9VZMHE1 {'2a02:c207:2008:5497::1'}
2016-12-06T08:58:3-/codemove/TYVRFD3NGGXK {'2a01:488:66:1000:5c33:8503:0:1'}
2016-12-06T08:58:3-/codemove/BRPB8Y32OAGA {'120.52.73.97'}
2016-12-06T08:58:3-/codemove/ICGPOQXLVXFS {'120.52.73.97'}
2016-12-06T08:58:3-/codemove/BJJHJB8J8T7C {'120.52.73.97'}
2016-12-06T08:58:3-/codemove/1N80W0N2R36C {'120.52.73.97'}
